## 1. Data loading

In [ ]:
from datasets import load_dataset

ds = load_dataset("moondream/glaucoma-detection")

print(ds)
print(ds["train"][0])
print(set(ds["train"]["class"]))

## 2. Number of images per set (before deduplication)

In [ ]:

print("=== Number of images per set ===")
n_train = len(ds["train"])
n_val = len(ds["validation"])
n_test = len(ds["test"])
n_total = n_train + n_val + n_test

print(f"Train:      {n_train:,} images")
print(f"Validation: {n_val:,} images")
print(f"Test:       {n_test:,} images")
print(f"Total:       {n_total:,} images")

print()
print(f"Ratio: train {n_train/n_total:.1%} / "
      f"val {n_val/n_total:.1%} / "
      f"test {n_test/n_total:.1%}")

## 3. Class distribution per set

In [ ]:
from collections import Counter

for set_name in ["train", "validation", "test"]:
    counts = Counter(ds[set_name]["class"])
    print(f"{set_name}: {dict(counts)}")

## 4. View sample images by class.

In [ ]:
classes = ["normal", "early", "advanced"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, cls in zip(axes, classes):
    idx = ds["train"]["class"].index(cls)
    img = Image.open(BytesIO(ds["train"][idx]["image"]["bytes"]))
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Check for corrupted images

In [ ]:
corrupted = []
for set_name in ["train", "validation", "test"]:
    for i, sample in enumerate(ds[set_name]):
        try:
            Image.open(BytesIO(sample["image"]["bytes"])).convert("RGB")
        except Exception as e:
            corrupted.append((set_name, i, str(e)))

print(f"Number of faulty images: {len(corrupted)}")

## 6. Check for duplicate images

In [ ]:
import hashlib

def hash_image(sample) -> str:
    return hashlib.md5(sample["image"]["bytes"]).hexdigest()


def analyze_set(hf_set, set_name: str):
    hash_to_indices = {}
    for i, sample in enumerate(hf_set):
        h = hash_image(sample)
        hash_to_indices.setdefault(h, []).append(i)

    duplicate_groups = {h: idxs for h, idxs in hash_to_indices.items() if len(idxs) > 1}
    n_duplicate_images = sum(len(idxs) - 1 for idxs in duplicate_groups.values())

    print(f"--- {set_name} ---")
    print(f"Total number of images: {len(hf_set)}")
    print(f"Number of unique images: {len(hash_to_indices)}")
    print(f"Number of duplicate image groups: {len(duplicate_groups)}")
    print(f"Excess photos due to duplication (will be removed if deduplicated): {n_duplicate_images}")
    print()

    return hash_to_indices, duplicate_groups


# ---- Internal duplicate checks for each set. ----
train_hashes, train_dups = analyze_set(ds["train"], "train")
val_hashes, val_dups = analyze_set(ds["validation"], "validation")
test_hashes, test_dups = analyze_set(ds["test"], "test")


# ---- Check for overlaps *between* sets (data leakage — a more serious issue). ----
leak_train_val = set(train_hashes) & set(val_hashes)
leak_train_test = set(train_hashes) & set(test_hashes)
leak_val_test = set(val_hashes) & set(test_hashes)

print("=== Check for leakage between sets. ===")
print(f"Overlapping images between training and validation sets: {len(leak_train_val)}")
print(f"Overlapping images between training and test sets: {len(leak_train_test)}")
print(f"Image overlap between validation and test sets: {len(leak_val_test)}")

if leak_train_val or leak_train_test or leak_val_test:
    print("\n[WARNING] There are duplicate images across sets — this is actual data leakage. "
          "Validation and test data are 'leaking' from the training set; this needs to be addressed before the results can be trusted.")
else:
    print("\nThere is no leakage between sets—only internal overlap (which is less severe)..")


# ---- (Optional) Preview a few duplicate images to visually confirm. ----
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

def show_duplicate_group(hf_set, indices, max_show=4):
    fig, axes = plt.subplots(1, min(len(indices), max_show), figsize=(4 * max_show, 4))
    if len(indices) == 1:
        axes = [axes]
    for ax, idx in zip(axes, indices[:max_show]):
        img = Image.open(BytesIO(hf_set[idx]["image"]["bytes"]))
        ax.imshow(img)
        ax.set_title(f"index {idx} -- class: {hf_set[idx]['class']}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

if train_dups:
    first_group = next(iter(train_dups.values()))
    print(f"\nFor example, a group of duplicate images in the training set. (index: {first_group}):")
    show_duplicate_group(ds["train"], first_group)

## 7. Deduplicated datasets

In [ ]:
import sys
sys.path.append("/content/DL2026-Group5-Project30")

from src.datasets.glaucoma_dataset import build_dataloaders

train_loader, val_loader, test_loader = build_dataloaders()

n_train_final = len(train_loader.dataset)
n_val_final = len(val_loader.dataset)
n_test_final = len(test_loader.dataset)
n_total_final = n_train_final + n_val_final + n_test_final

print("=== FINAL dataset ===")
print(f"Train:      {n_train_final:,} images")
print(f"Validation: {n_val_final:,} images")
print(f"Test:       {n_test_final:,} images")
print(f"Tổng:       {n_total_final:,} images")

print()
print("=== Comparison before and after deduplication ===")
print(f"Train:      {n_train:,} -> {n_train_final:,}  (decrease {n_train - n_train_final:,})")
print(f"Validation: {n_val:,} -> {n_val_final:,}  (decrease {n_val - n_val_final:,})")
print(f"Test:       {n_test:,} -> {n_test_final:,}  (decrease {n_test - n_test_final:,})")